# 05 · Agentic RAG
### Practical LLM Engineering — Module 04: RAG Systems

> **What you'll learn**
> - Limitations of single-step RAG and when agentic loops are needed
> - Iterative retrieval: retrieve → reason → decide whether to retrieve again
> - Query routing: selecting the right knowledge source per query
> - Self-RAG: the model decides when to retrieve and how to critique its own answer
> - Tool-augmented RAG: calculator, code executor, web search
> - Engineering: termination conditions, loop safety, cost control

---


## 1. Overview

Single-step RAG retrieves once and generates once. It fails on:
- **Multi-hop questions** — "Who invented the algorithm used in Elasticsearch's ranking?"
- **Ambiguous questions** — need clarification before retrieval
- **Calculation-heavy answers** — retrieval finds data but can't compute
- **Iterative refinement** — first answer reveals what else to look up

**Agentic RAG** gives the model agency over the retrieval loop:

```
Query
  ↓
Plan: what do I need to answer this?
  ↓
Retrieve (tool call)
  ↓
Reason: do I have enough? If not, retrieve again
  ↓
[optional: use calculator / code tool]
  ↓
Generate final answer
  ↓
Self-critique: is my answer grounded and complete?
```


## 2. Setup

In [ ]:
# !pip install numpy matplotlib -q
# ── Shared utilities (carried from Module 03) ────────────────────────
import os, re, json, time, math, random, textwrap
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict
from sklearn.preprocessing import normalize

# ── Mock embedder ─────────────────────────────────────────────────────
class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

# ── BM25 ──────────────────────────────────────────────────────────────
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self._corpus, self._doc_freqs, self._avgdl, self._N = [], defaultdict(int), 0.0, 0
    def _tok(self, t): return re.findall(r"\b\w+\b", t.lower())
    def fit(self, docs):
        self._corpus = [self._tok(d) for d in docs]; self._N = len(self._corpus)
        self._avgdl  = np.mean([len(d) for d in self._corpus])
        for doc in self._corpus:
            for t in set(doc): self._doc_freqs[t] += 1
    def _idf(self, t):
        n = self._doc_freqs.get(t, 0)
        return math.log((self._N - n + 0.5) / (n + 0.5) + 1)
    def get_scores(self, query):
        qt = self._tok(query)
        scores = []
        for doc in self._corpus:
            dl = len(doc); tf = defaultdict(int)
            for t in doc: tf[t] += 1
            s = sum(self._idf(t) * tf[t] * (self.k1+1) /
                    (tf[t] + self.k1*(1-self.b+self.b*dl/self._avgdl))
                    for t in qt if t in tf)
            scores.append(s)
        return np.array(scores)
    def get_top_k(self, query, k=5):
        s = self.get_scores(query); idx = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in idx]

# ── Vector store ──────────────────────────────────────────────────────
@dataclass
class Doc:
    id: str; text: str; metadata: dict = field(default_factory=dict)

class VectorStore:
    def __init__(self, embedder, dim=64):
        self.embedder = embedder
        self._vecs: np.ndarray = np.zeros((0, dim), dtype=np.float32)
        self._docs: list[Doc]  = []
    def add(self, docs: list[Doc]):
        vecs = self.embedder.embed([d.text for d in docs])
        self._vecs = np.vstack([self._vecs, vecs]) if len(self._vecs) else vecs
        self._docs.extend(docs)
    def search(self, query: str, k=5, where=None) -> list[tuple[Doc, float]]:
        if not self._docs: return []
        q = self.embedder.embed([query])[0]
        scores = self._vecs @ q
        if where:
            for i, doc in enumerate(self._docs):
                for key, val in where.items():
                    if doc.metadata.get(key) != val: scores[i] = -999
        top = np.argsort(-scores)[:k]
        return [(self._docs[i], float(scores[i])) for i in top]
    def __len__(self): return len(self._docs)

# ── Mock LLM ──────────────────────────────────────────────────────────
@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    """Returns templated answers grounded in provided context."""
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, system: str, user: str,
                 max_tokens=512, temperature=0.0) -> LLMResponse:
        time.sleep(0.02)
        ctx_match = re.search(r"Context:(.*?)(?:Question:|$)", user, re.DOTALL)
        ctx = ctx_match.group(1).strip()[:200] if ctx_match else ""
        q   = re.search(r"Question:(.*?)$", user, re.DOTALL)
        q   = q.group(1).strip()[:80] if q else user[:80]
        answer = (f"Based on the provided context, {q.lower().rstrip('?')} "
                  f"can be understood as follows: {ctx[:120]}... "
                  f"This answer is grounded in the retrieved documents.")
        n_in  = len((system+user).split())
        n_out = len(answer.split())
        return LLMResponse(answer, n_in, n_out, 45.0)
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

embedder = MockEmbedder(dim=64)
llm      = MockLLM()
print("✅ Shared utilities loaded (embedder + BM25 + VectorStore + MockLLM)")


In [ ]:
# Multi-source knowledge base
SOURCES = {
    "ml_docs": [
        Doc("ml001", "Transformers use scaled dot-product attention: Q·K^T/sqrt(d_k).", {"source":"ml_docs"}),
        Doc("ml002", "BERT is pre-trained with masked LM and next-sentence prediction.",  {"source":"ml_docs"}),
        Doc("ml003", "LoRA adds low-rank adapter matrices to fine-tune LLMs efficiently.",{"source":"ml_docs"}),
        Doc("ml004", "Gradient descent minimises loss by moving in the negative gradient direction.", {"source":"ml_docs"}),
    ],
    "retrieval_docs": [
        Doc("re001", "BM25 ranks by TF-IDF with length normalisation; used in Lucene/Elasticsearch.", {"source":"retrieval_docs"}),
        Doc("re002", "FAISS provides GPU-accelerated ANN search using IVF and HNSW indexes.", {"source":"retrieval_docs"}),
        Doc("re003", "Hybrid search merges dense and sparse retrieval with RRF or weighted fusion.", {"source":"retrieval_docs"}),
        Doc("re004", "Reranking uses cross-encoders to improve precision after first-stage retrieval.", {"source":"retrieval_docs"}),
    ],
    "rag_docs": [
        Doc("rg001", "RAG grounds LLM outputs in retrieved documents to reduce hallucination.",  {"source":"rag_docs"}),
        Doc("rg002", "Chunking splits documents before embedding; 256-512 tokens is typical.",    {"source":"rag_docs"}),
        Doc("rg003", "Advanced RAG includes query transformation, compression, and reranking.",   {"source":"rag_docs"}),
        Doc("rg004", "Agentic RAG lets the model decide when and what to retrieve iteratively.",  {"source":"rag_docs"}),
    ],
}

stores = {}
for name, docs in SOURCES.items():
    stores[name] = VectorStore(embedder)
    stores[name].add(docs)

all_docs = [d for docs in SOURCES.values() for d in docs]
global_store = VectorStore(embedder); global_store.add(all_docs)
print(f"Multi-source KB: {sum(len(d) for d in SOURCES.values())} docs across {len(SOURCES)} sources")


## 3. Query Routing

In [ ]:
class QueryRouter:
    """
    Route a query to the most relevant knowledge source.
    In production: use a fine-tuned classifier or embedding similarity to source centroids.
    """
    SOURCE_KEYWORDS = {
        "ml_docs":        ["transformer", "bert", "lora", "gradient", "attention",
                            "training", "neural", "fine-tune", "model"],
        "retrieval_docs": ["bm25", "faiss", "hnsw", "ivf", "index", "annoy",
                            "search", "elasticsearch", "approximate"],
        "rag_docs":       ["rag", "retrieval augmented", "chunk", "chunk", "pipeline",
                            "grounding", "hallucination", "citation", "agentic"],
    }

    def __init__(self, embedder, stores: dict[str, VectorStore]):
        self.embedder = embedder
        self.stores   = stores
        # Build source centroid embeddings
        self._centroids = {}
        for source, keywords in self.SOURCE_KEYWORDS.items():
            vecs = embedder.embed(keywords)
            self._centroids[source] = vecs.mean(axis=0)
            self._centroids[source] /= np.linalg.norm(self._centroids[source]) + 1e-9

    def route(self, query: str, top_n: int = 1) -> list[str]:
        """Return the top_n most relevant source names for this query."""
        q_vec = self.embedder.embed([query])[0]
        scores = {src: float(np.dot(q_vec, centroid))
                  for src, centroid in self._centroids.items()}
        return sorted(scores, key=lambda x: -scores[x])[:top_n]

    def retrieve(self, query: str, k: int = 3) -> list[tuple[Doc, float]]:
        sources = self.route(query, top_n=2)
        results = []
        for src in sources:
            results.extend(self.stores[src].search(query, k=k))
        results.sort(key=lambda x: -x[1])
        return results[:k]


router = QueryRouter(embedder, stores)
test_queries = [
    "how does BM25 rank documents",
    "what is LoRA fine-tuning",
    "how does agentic RAG work",
    "what is FAISS and when should I use it",
]

print("Query routing decisions:")
for q in test_queries:
    routed  = router.route(q, top_n=2)
    results = router.retrieve(q, k=2)
    print(f"  Q: {q[:50]}")
    print(f"     Routes to: {routed}")
    print(f"     Top doc: [{results[0][0].id}] {results[0][0].text[:55]}")
    print()


## 4. Iterative Retrieval

In [ ]:
@dataclass
class AgentStep:
    action:    str   # "retrieve" | "generate" | "done"
    query:     str
    retrieved: list = field(default_factory=list)
    reasoning: str  = ""
    answer:    str  = ""


class IterativeRAGAgent:
    """
    Agent that retrieves, reasons, and decides whether more retrieval is needed.
    Max iterations prevents infinite loops.
    """
    def __init__(self, store: VectorStore, llm: MockLLM,
                 max_iter: int = 3, k: int = 3):
        self.store    = store
        self.llm      = llm
        self.max_iter = max_iter
        self.k        = k

    def _should_retrieve_more(self, question: str, context: str,
                                draft_answer: str) -> bool:
        """Decide if the current context is sufficient to answer the question."""
        system = "You are a retrieval planner."
        user   = (f"Question: {question}\n\n"
                  f"Context so far: {context[:200]}\n\n"
                  f"Draft answer: {draft_answer[:100]}\n\n"
                  f"Is this context sufficient? Reply YES or NO.")
        resp   = self.llm(system, user, max_tokens=10)
        return "no" in resp.text.lower()

    def _refine_query(self, question: str, prev_queries: list[str],
                       draft_answer: str) -> str:
        """Generate a follow-up retrieval query based on what is still missing."""
        system = "You are a search assistant."
        user   = (f"Original question: {question}\n"
                  f"Already searched: {prev_queries}\n"
                  f"Draft answer so far: {draft_answer[:100]}\n\n"
                  f"Write a follow-up search query to fill the missing information.")
        resp   = self.llm(system, user, max_tokens=40)
        return resp.text.strip()

    def run(self, question: str) -> dict:
        steps:        list[AgentStep] = []
        all_context:  list[str]       = []
        prev_queries: list[str]       = []
        draft_answer  = ""

        for iteration in range(self.max_iter):
            # Retrieve
            query = question if not prev_queries else self._refine_query(
                question, prev_queries, draft_answer)
            prev_queries.append(query)

            retrieved = self.store.search(query, k=self.k)
            context   = "\n".join(f"[{doc.id}] {doc.text}" for doc, _ in retrieved)
            all_context.append(context)

            step = AgentStep(action="retrieve", query=query,
                             retrieved=[doc.id for doc, _ in retrieved])
            steps.append(step)

            # Generate draft
            full_ctx = "\n".join(all_context)
            system   = "You are a factual assistant. Answer using only the context."
            user     = f"Context:\n{full_ctx}\n\nQuestion: {question}\nAnswer:"
            resp     = self.llm(system, user, max_tokens=200)
            draft_answer = resp.text

            # Decide: done or retrieve more?
            if not self._should_retrieve_more(question, full_ctx, draft_answer):
                break

        steps.append(AgentStep(action="done", query="", answer=draft_answer))
        return {
            "question":   question,
            "answer":     draft_answer,
            "steps":      steps,
            "n_iter":     len([s for s in steps if s.action == "retrieve"]),
            "total_docs": sum(len(s.retrieved) for s in steps if s.action == "retrieve"),
        }


agent = IterativeRAGAgent(global_store, llm, max_iter=3, k=3)

MULTI_HOP_QUESTIONS = [
    "How does BM25 differ from transformer-based dense retrieval, and which should I use for RAG?",
    "What techniques does advanced RAG add on top of basic RAG?",
]

print("Iterative RAG Agent\n")
for q in MULTI_HOP_QUESTIONS:
    result = agent.run(q)
    print(f"Q: {q}")
    print(f"Iterations: {result['n_iter']}  |  Total docs retrieved: {result['total_docs']}")
    for s in result["steps"]:
        if s.action == "retrieve":
            print(f"  [retrieve] query={s.query[:40]}  docs={s.retrieved}")
    print(f"  [answer] {result['answer'][:100]}...")
    print()


## 5. Self-RAG: Critique and Refinement

In [ ]:
class SelfRAG:
    """
    Self-RAG: LLM decides whether to retrieve, generates, then self-critiques.
    Inspired by: Asai et al. (2023) Self-RAG: Learning to Retrieve, Generate, Reflect.
    Special tokens simulated: [Retrieve], [IsRel], [IsSup], [IsUse]
    """
    def __init__(self, store: VectorStore, llm: MockLLM, k: int = 3):
        self.store = store
        self.llm   = llm
        self.k     = k

    def _should_retrieve(self, question: str) -> bool:
        """[Retrieve] token: decide if retrieval is needed."""
        system = "You are a retrieval decision assistant."
        user   = (f"Does answering this question require external knowledge? "
                  f"Reply YES or NO.\n\nQuestion: {question}")
        resp   = self.llm(system, user, max_tokens=5)
        return "yes" in resp.text.lower() or True  # default True for demo

    def _is_relevant(self, question: str, doc_text: str) -> float:
        """[IsRel] token: score relevance of retrieved document."""
        system = "You are a relevance scorer."
        user   = (f"Is this document relevant to the question? "
                  f"Score 0.0 to 1.0.\nQuestion: {question}\nDoc: {doc_text}")
        resp   = self.llm(system, user, max_tokens=5)
        nums   = re.findall(r"0?\.[0-9]+|1\.0|1|0", resp.text)
        return float(nums[0]) if nums else 0.5

    def _is_supported(self, answer: str, context: str) -> float:
        """[IsSup] token: is the answer fully supported by context?"""
        system = "You are a fact-checker."
        user   = (f"Is the following answer fully supported by the context? "
                  f"Score 0.0 to 1.0.\nContext: {context[:200]}\nAnswer: {answer[:100]}")
        resp   = self.llm(system, user, max_tokens=5)
        nums   = re.findall(r"0?\.[0-9]+|1\.0|1|0", resp.text)
        return float(nums[0]) if nums else 0.5

    def _is_useful(self, question: str, answer: str) -> float:
        """[IsUse] token: does the answer address the question?"""
        system = "You are a relevance evaluator."
        user   = (f"Does this answer address the question? "
                  f"Score 0.0 to 1.0.\nQuestion: {question}\nAnswer: {answer[:100]}")
        resp   = self.llm(system, user, max_tokens=5)
        nums   = re.findall(r"0?\.[0-9]+|1\.0|1|0", resp.text)
        return float(nums[0]) if nums else 0.5

    def run(self, question: str) -> dict:
        # Step 1: Decide whether to retrieve
        retrieve = self._should_retrieve(question)
        context, retrieved_ids = "", []

        if retrieve:
            docs = self.store.search(question, k=self.k)
            # Filter by relevance
            relevant = [(doc, sc) for doc, sc in docs
                        if self._is_relevant(question, doc.text) >= 0.4]
            context      = "\n".join(f"[{doc.id}] {doc.text}" for doc, _ in relevant)
            retrieved_ids = [doc.id for doc, _ in relevant]

        # Step 2: Generate
        system = "You are a factual assistant."
        user   = (f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
                   if context else f"Question: {question}\nAnswer:")
        resp   = self.llm(system, user, max_tokens=200)
        answer = resp.text

        # Step 3: Self-critique
        is_sup  = self._is_supported(answer, context) if context else 1.0
        is_use  = self._is_useful(question, answer)
        overall = (is_sup + is_use) / 2

        return {
            "question":     question,
            "answer":       answer,
            "retrieved":    retrieved_ids,
            "is_supported": is_sup,
            "is_useful":    is_use,
            "overall":      overall,
            "retrieved_flag": retrieve,
        }


self_rag = SelfRAG(global_store, llm)

print("Self-RAG with self-critique tokens\n")
print(f"{'Question':<45} {'Retrieved':>10} {'IsSup':>7} {'IsUse':>7} {'Overall':>9}")
print("-" * 82)
for q in ["How does RAG reduce hallucination?",
           "What is 2+2?",
           "Compare BM25 and hybrid search for RAG."]:
    r = self_rag.run(q)
    print(f"{q[:43]:<45} {str(r['retrieved']):<10} {r['is_supported']:>7.2f} "
          f"{r['is_useful']:>7.2f} {r['overall']:>9.2f}")


## 6. Tool-Augmented RAG

In [ ]:
# ── Tool registry ────────────────────────────────────────────────────
import math as _math

TOOLS = {
    "calculator": {
        "description": "Evaluate a mathematical expression. Input: Python expression string.",
        "fn": lambda expr: str(eval(expr, {"__builtins__": {}},
                                    {k: getattr(_math, k) for k in dir(_math) if not k.startswith("_")}))
    },
    "retrieve": {
        "description": "Search the knowledge base. Input: search query string.",
        "fn": lambda q: "\n".join(f"[{doc.id}] {doc.text[:80]}"
                                    for doc, _ in global_store.search(q, k=2))
    },
    "count_words": {
        "description": "Count words in a string. Input: text string.",
        "fn": lambda t: str(len(t.split()))
    },
}


class ToolAugmentedRAG:
    """RAG agent that can call external tools during generation."""

    def __init__(self, llm: MockLLM, tools: dict, max_steps: int = 4):
        self.llm       = llm
        self.tools     = tools
        self.max_steps = max_steps

    def _plan_tool_use(self, question: str, history: list[dict]) -> Optional[dict]:
        """Decide if a tool should be called next."""
        tool_descs = "\n".join(f"- {name}: {t['description']}"
                                 for name, t in self.tools.items())
        hist_str   = "\n".join(f"Tool: {h['tool']} | Input: {h['input']} | Output: {h['output']}"
                                  for h in history)
        system     = "You are an agent that decides when to use tools."
        user       = (f"Question: {question}\n\nTools available:\n{tool_descs}\n\n"
                      f"Steps so far:\n{hist_str or 'None'}\n\n"
                      f"Should a tool be called? If YES, respond: TOOL: <name> INPUT: <input>. "
                      f"If NO, respond: ANSWER")
        resp = self.llm(system, user, max_tokens=80)

        if resp.text.strip().upper().startswith("ANSWER"):
            return None
        m = re.search(r"TOOL:\s*(\w+)\s+INPUT:\s*(.*)", resp.text, re.IGNORECASE)
        if m:
            return {"tool": m.group(1).strip(), "input": m.group(2).strip()}
        return None

    def run(self, question: str) -> dict:
        history = []
        for step in range(self.max_steps):
            action = self._plan_tool_use(question, history)
            if action is None:
                break
            tool_name = action["tool"]
            if tool_name not in self.tools:
                break
            try:
                output = self.tools[tool_name]["fn"](action["input"])
            except Exception as e:
                output = f"Error: {e}"
            history.append({**action, "output": output})

        # Final generation
        ctx    = "\n".join(f"[{h['tool']}({h['input']})] → {h['output']}"
                             for h in history)
        system = "You are a helpful assistant. Use the tool results to answer."
        user   = f"Tool results:\n{ctx}\n\nQuestion: {question}\nAnswer:"
        resp   = self.llm(system, user, max_tokens=200)

        return {"question": question, "answer": resp.text,
                "steps": history, "n_tool_calls": len(history)}


tool_rag = ToolAugmentedRAG(llm, TOOLS, max_steps=3)

print("Tool-Augmented RAG\n")
tool_queries = [
    "What is sqrt(144) + 17?",
    "How does hybrid search work in RAG systems?",
    "If I have 256 embedding dimensions and 1 million documents, how many GB is the index?",
]

for q in tool_queries:
    r = tool_rag.run(q)
    print(f"Q: {q}")
    for s in r["steps"]:
        print(f"  [{s['tool']}({s['input'][:30]})] → {s['output'][:50]}")
    print(f"  A: {r['answer'][:100]}...")
    print()


## 7. Key Takeaways

| Concept | Summary |
|---|---|
| **Query routing** | Direct queries to the most relevant knowledge source |
| **Iterative retrieval** | Retrieve → draft → assess → retrieve again if needed |
| **Self-RAG** | Model decides when to retrieve and critiques its own output |
| **IsRel/IsSup/IsUse** | Self-RAG critique tokens for relevance, support, usefulness |
| **Tool-augmented RAG** | Add calculator, code executor, or web search as tools |
| **Termination** | Always set max_iterations to prevent infinite loops |
| **Cost control** | Each extra retrieval = extra LLM calls; budget per query carefully |

---

🎉 **Module 04: RAG Systems — Complete!**

All 5 notebooks:
- `01_basic_rag.ipynb` ✅
- `02_advanced_rag_pipeline.ipynb` ✅
- `03_reranking.ipynb` ✅
- `04_rag_evaluation.ipynb` ✅
- `05_agentic_rag.ipynb` ✅

**Next module →** `05_llm_agents/` — function calling, tool use, and multi-step agents.
